### functionCall-Demo2

In [15]:
from langchain_core.messages import  ToolMessage
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
import json,requests
from tools import *
load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("DASHSCOPE_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("BAILIAN_BASE_URL")
print(os.getenv("OPENAI_BASE_URL"))
init_model = init_chat_model(model_provider="openai", model="qwen3.5-flash", api_key=os.getenv("OPENAI_API_KEY"),
                        base_url=os.getenv("OPENAI_BASE_URL"), )
model = init_model.bind_tools(tools)

def get_current_weather(location:str):
    """获取制定城市的当前天气"""
    with open(file="./city_code.json",encoding=u"utf-8") as f:
        data = json.load(f)
    city_code = None
    #匹配城市编码
    for loc in data:
        if location == loc["city"]:
            city_code = loc["code"]
            break
            
    if city_code:
        weather_url = f"http://t.weather.itboy.net/api/weather/city/{city_code}"
        response = requests.get(weather_url)
        result = response.json()
        forecast = result["data"]["forecast"][0]
        weather_info = {
            "location": location,
            "high_temperature": forecast["high"],
            "low_temperature": forecast["low"],
            "week": forecast["week"],
            "type": forecast["type"],
        }
        return json.dumps(weather_info, ensure_ascii=False)
    else:
        return json.dumps({"error": "城市不存在"}, ensure_ascii=False)
        
def main():
    """主函数"""
    messages = [
        {"role": "system", "content": "你是一个天气播报小助手，你需要根据用户提供的地址来回答当地的天气情况。"},
        {"role": "user", "content": "今天北京的天气如何"}
    ]
    model_response = model.invoke(messages)
    print(model_response)
    tool = model_response.tool_calls[0]
    print(tool)
    function_name = tool['name']
    args = tool['args']
    tool_id = tool['id']
    function_result = ""
    if function_name == "get_current_weather":
        function_result = get_current_weather(args['location'])
    messages.append(model_response)
    messages.append(ToolMessage(
        tool_call_id= tool_id,
        content = function_result,
    ))
    final_response = model.invoke(messages)
    print(final_response.content)

if __name__ == "__main__":
    main()

https://dashscope.aliyuncs.com/compatible-mode/v1
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 303, 'total_tokens': 367, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 33, 'rejected_prediction_tokens': None, 'text_tokens': 64}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 303}}, 'model_provider': 'openai', 'model_name': 'qwen3.5-flash', 'system_fingerprint': None, 'id': 'chatcmpl-65343b92-b130-94bc-bde4-ca5893021782', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019e4639-bdae-7493-a3e4-c7ea3f359e46-0' tool_calls=[{'name': 'get_current_weather', 'args': {'location': '北京'}, 'id': 'call_78df3e67197f4c91a6d73cab', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 303, 'output_tokens': 64, 'total_tokens': 367, 'input_token_details': {}, 'output_token_details': {'reasoning